# Experimento 3. RoPE y coste computacional

## 1. Configuración
Prepararemos PyTorch, Pandas y Matplotlib. Usaremos FP32 en la prueba analítica y FP16 para medir rendimiento en CUDA (FP32 si solo tenemos CPU). Guardaremos tablas y figuras en `resultados_exp3`, sobrescribiendo los archivos al repetir la ejecución.

In [ ]:
import time
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import display

# Abriremos el notebook desde la raíz del proyecto o desde su propia carpeta.
raiz = Path.cwd()
if not (raiz / 'minigpt').is_dir():
    raiz = raiz.parent
sys.path.insert(0, str(raiz))

from minigpt.config import GPTConfig
from minigpt.model import GPTModel
from minigpt.rope import RoPECache, apply_rotary

torch.manual_seed(42)
dispositivo = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
precision = torch.float16 if dispositivo.type == 'cuda' else torch.float32
nombre_precision = 'FP16' if precision == torch.float16 else 'FP32'

carpeta_experimentos = raiz / 'experimentos_2_y_3'
resultados = carpeta_experimentos / 'resultados_exp3'
resultados.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize': (7, 4), 'font.size': 11, 'savefig.dpi': 300})

def sincronizar():
    if dispositivo.type == 'cuda':
        torch.cuda.synchronize(dispositivo)

print(f'Dispositivo: {dispositivo} | Precisión: {precision}')
if dispositivo.type == 'cuda':
    print(torch.cuda.get_device_name(dispositivo))
else:
    print('Mediremos tiempos en CPU; la VRAM no estará disponible.')
print(f'Guardaremos los resultados en: {resultados}')

## 2. Carga de los modelos entrenados
Cargaremos los modelos independientes Learned y RoPE entrenados en el experimento 2, con sus configuraciones originales. Ajustaremos las rutas a los checkpoints guardados.

In [ ]:
# Esperaremos model_config y model_state_dict, como en minigpt/finetune.py.
rutas = {
    'Learned': carpeta_experimentos / 'pesos' / 'learned' / 'best.pt',
    'RoPE': carpeta_experimentos / 'pesos' / 'rope' / 'best.pt',
}
faltantes = [str(ruta) for ruta in rutas.values() if not ruta.is_file()]
if faltantes:
    raise FileNotFoundError(
        'Faltan checkpoints entrenados del experimento 2: ' + ', '.join(faltantes)
        + '. Completa ese experimento y ajusta las rutas. No se usarán modelos aleatorios.'
    )

modelos = {}
configuraciones = {}
for nombre, ruta in rutas.items():
    checkpoint = torch.load(ruta, map_location='cpu', weights_only=True)
    if not isinstance(checkpoint, dict) or not {'model_config', 'model_state_dict'} <= checkpoint.keys():
        raise ValueError(f'{ruta}: necesitamos model_config y model_state_dict completos.')
    configuracion = GPTConfig(**checkpoint['model_config'])
    esquema = 'learned' if nombre == 'Learned' else 'rope'
    if configuracion.positional != esquema:
        raise ValueError(f'{ruta}: la configuración no corresponde al modelo {nombre}.')
    modelo = GPTModel(configuracion)
    # La carga estricta sustituye todos los parámetros antes de medir.
    modelo.load_state_dict(checkpoint['model_state_dict'], strict=True)
    modelos[nombre] = modelo.eval()
    configuraciones[nombre] = configuracion
    del checkpoint

config_learned = vars(configuraciones['Learned'])
config_rope = vars(configuraciones['RoPE'])
if any(config_learned[campo] != config_rope[campo]
       for campo in config_learned if campo != 'positional'):
    raise ValueError('Los checkpoints deben tener la misma configuración salvo positional.')

tabla_configuracion = pd.DataFrame({nombre: vars(cfg) for nombre, cfg in configuraciones.items()})
display(tabla_configuracion)
tabla_configuracion.to_csv(resultados / 'configuracion.csv')
pd.Series({nombre: str(ruta) for nombre, ruta in rutas.items()}, name='checkpoint').to_csv(
    resultados / 'checkpoints.csv')

## 3. Validación analítica de RoPE
Ahora comprobaremos que, manteniendo Q y K fijos, desplazar sus dos posiciones no cambia el producto escalar si conservamos la separación:

$$ (R_m q)^\top (R_n k) = q^\top R_{n-m} k. $$

Usaremos los mismos 16 pares Q/K normalizados en distintas posiciones y compararemos con `(0, separación)`. Aplicaremos `RoPECache` y `apply_rotary` del proyecto en CPU y FP32, la precisión original de sus tablas trigonométricas.

In [ ]:
cfg = configuraciones['RoPE']
generador = torch.Generator().manual_seed(42)
q = torch.randn(16, 1, 1, cfg.head_dim, generator=generador, dtype=torch.float32)
k = torch.randn(16, 1, 1, cfg.head_dim, generator=generador, dtype=torch.float32)
q = torch.nn.functional.normalize(q, dim=-1)
k = torch.nn.functional.normalize(k, dim=-1)
separaciones = [0, 1, 4, 16, 64, 128]
desplazamientos = [0, 1, 8, 32, 128, 256, 512]
cache = RoPECache(cfg.head_dim, max(desplazamientos) + max(separaciones) + 1, cfg.rope_theta)

filas_rope = []
with torch.inference_mode():
    for separacion in separaciones:
        coseno, seno = cache.get(1, offset=separacion)
        referencia = (q * apply_rotary(k, coseno, seno)).sum(dim=-1)
        for desplazamiento in desplazamientos:
            coseno, seno = cache.get(1, offset=desplazamiento)
            q_rotada = apply_rotary(q, coseno, seno)
            coseno, seno = cache.get(1, offset=desplazamiento + separacion)
            k_rotada = apply_rotary(k, coseno, seno)
            producto = (q_rotada * k_rotada).sum(dim=-1)
            error = (producto - referencia).abs()
            filas_rope.append({
                'separacion': separacion, 'desplazamiento': desplazamiento,
                'producto_par_0': producto[0].item(),
                'error_medio': error.mean().item(), 'error_maximo': error.max().item(),
            })

tabla_rope = pd.DataFrame(filas_rope)
tabla_rope.to_csv(resultados / 'validacion_rope.csv', index=False)
display(tabla_rope.groupby('separacion').agg(
    error_medio=('error_medio', 'mean'), error_maximo=('error_maximo', 'max')))

figura, ejes = plt.subplots(1, 2, figsize=(12, 4))
for separacion, grupo in tabla_rope.groupby('separacion'):
    ejes[0].plot(grupo['desplazamiento'], grupo['producto_par_0'], marker='o',
                 markersize=3, label=f'Δ = {separacion}')
    ejes[1].plot(grupo['desplazamiento'], grupo['error_maximo'], marker='o',
                 markersize=3, label=f'Δ = {separacion}')
ejes[0].set(title='Producto escalar del mismo par Q/K',
            xlabel='Desplazamiento absoluto (posiciones)', ylabel='Producto escalar (sin unidades)')
ejes[1].set(title='Error máximo entre los 16 pares',
            xlabel='Desplazamiento absoluto (posiciones)', ylabel='Error absoluto (sin unidades)')
for eje in ejes:
    eje.grid(alpha=0.25)
    eje.legend(fontsize=8)
figura.tight_layout()
figura.savefig(resultados / 'invariancia_relativa.png')
figura.savefig(resultados / 'invariancia_relativa.pdf')
plt.show()
print(f"Error absoluto máximo: {tabla_rope['error_maximo'].max():.3e}")

Interpretaremos estos errores como una comprobación con Q y K fijos, no como una prueba de invariancia del modelo completo al cambiar el contexto.

## 4. Comparación computacional
Primero contaremos los parámetros totales y los posicionales de cada modelo.

In [ ]:
tabla_parametros = pd.DataFrame([
    {'modelo': nombre, 'parametros_totales': sum(p.numel() for p in modelo.parameters()),
     'parametros_posicionales': modelo.positional_parameters}
    for nombre, modelo in modelos.items()
])
display(tabla_parametros)
tabla_parametros.to_csv(resultados / 'parametros.csv', index=False)

Ahora mediremos el forward completo, incluidos los logits, en evaluación y sin gradientes. Usaremos lote 1, las mismas entradas y FP16 en CUDA (FP32 en CPU). Los IDs sintéticos servirán solo como carga de trabajo: no mediremos calidad lingüística ni generación token a token.

Para cada longitud haremos 5 forwards de calentamiento y 5 mediciones de 10 forwards, sincronizando CUDA. Dejaremos fuera del tiempo la carga y las transferencias; mantendremos solo un modelo en GPU.

Mediremos la memoria asignada por PyTorch: **base** con el modelo en GPU y sin entrada, **pico** durante un forward adicional e **incremento = pico − base** (incluye la entrada y los temporales). No será toda la VRAM de `nvidia-smi`; en CPU no tendremos estos datos.

In [ ]:
lote = 1
calentamiento = 5
repeticiones = 5
iteraciones = 10
longitudes = sorted({min(n, cfg.context_length) for n in [64, 128, 256, 512, 1024]})
generador = torch.Generator().manual_seed(43)
entradas_cpu = torch.randint(cfg.vocab_size, (lote, max(longitudes)), generator=generador)
mediciones = []

with torch.inference_mode():
    for nombre, modelo in modelos.items():
        modelo.to(device=dispositivo, dtype=precision)
        sincronizar()
        base_mib = (torch.cuda.memory_allocated(dispositivo) / 2**20
                    if dispositivo.type == 'cuda' else float('nan'))
        for longitud in longitudes:
            entrada = entradas_cpu[:, :longitud].contiguous().to(dispositivo)
            for _ in range(calentamiento):
                salida = modelo(entrada)
                if _ == calentamiento - 1 and not torch.isfinite(salida).all().item():
                    raise ValueError('Salida no finita: cambia precision a torch.float32 para ambos modelos.')
                del salida
            sincronizar()

            for repeticion in range(repeticiones):
                sincronizar()
                inicio = time.perf_counter()
                for _ in range(iteraciones):
                    modelo(entrada)
                sincronizar()
                segundos = time.perf_counter() - inicio

                pico_mib = incremento_mib = float('nan')
                if dispositivo.type == 'cuda':
                    torch.cuda.reset_peak_memory_stats(dispositivo)
                    modelo(entrada)
                    sincronizar()
                    pico = torch.cuda.max_memory_allocated(dispositivo)
                    pico_mib = pico / 2**20
                    incremento_mib = pico_mib - base_mib

                mediciones.append({
                    'modelo': nombre, 'longitud': longitud, 'repeticion': repeticion + 1,
                    'tiempo_ms': segundos / iteraciones * 1000,
                    'tokens_s': lote * longitud * iteraciones / segundos,
                    'base_vram_mib': base_mib, 'pico_vram_mib': pico_mib,
                    'incremento_vram_mib': incremento_mib,
                })
            del entrada
        modelo.to('cpu')
        if dispositivo.type == 'cuda':
            torch.cuda.empty_cache()
        print(f'{nombre}: mediciones terminadas.')

tabla_mediciones = pd.DataFrame(mediciones)
tabla_mediciones.to_csv(resultados / 'mediciones.csv', index=False)
resumen = tabla_mediciones.groupby(['modelo', 'longitud']).agg(
    tiempo_ms=('tiempo_ms', 'mean'), tiempo_std_ms=('tiempo_ms', 'std'),
    tokens_s=('tokens_s', 'mean'), tokens_std_s=('tokens_s', 'std'),
    base_vram_mib=('base_vram_mib', 'mean'), pico_vram_mib=('pico_vram_mib', 'mean'),
    incremento_vram_mib=('incremento_vram_mib', 'mean'),
).reset_index()
display(resumen[['modelo', 'longitud', 'tiempo_ms', 'tiempo_std_ms', 'tokens_s', 'tokens_std_s']].round(3))
resumen.to_csv(resultados / 'resumen.csv', index=False)
pd.Series({'dispositivo': str(dispositivo), 'precision': str(precision), 'lote': lote,
           'calentamiento': calentamiento, 'repeticiones': repeticiones,
           'iteraciones': iteraciones}, name='valor').to_csv(resultados / 'condiciones.csv')

Representaremos la velocidad media con su desviación estándar. Después veremos la base, el pico y el incremento de memoria; la altura total de cada barra será el pico del forward.

In [ ]:
figura, eje = plt.subplots()
for nombre, grupo in resumen.groupby('modelo'):
    eje.errorbar(grupo['longitud'], grupo['tokens_s'], yerr=grupo['tokens_std_s'],
                 marker='o', capsize=4, label=nombre)
eje.set(title=f'Velocidad del forward completo ({dispositivo.type.upper()}, {nombre_precision})',
        xlabel='Longitud de secuencia (tokens)', ylabel='Rendimiento (tokens/s)')
eje.set_xticks(longitudes)
eje.grid(alpha=0.25)
eje.legend()
figura.tight_layout()
figura.savefig(resultados / 'velocidad.png')
figura.savefig(resultados / 'velocidad.pdf')
plt.show()

if dispositivo.type == 'cuda':
    display(resumen[['modelo', 'longitud', 'base_vram_mib', 'pico_vram_mib',
                     'incremento_vram_mib']].round(2))
    figura, ejes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
    for eje, (nombre, grupo) in zip(ejes, resumen.groupby('modelo')):
        posiciones = grupo['longitud'].astype(str)
        eje.bar(posiciones, grupo['base_vram_mib'], color='lightgray', label='Base del modelo')
        eje.bar(posiciones, grupo['incremento_vram_mib'], bottom=grupo['base_vram_mib'],
                color='steelblue', label='Incremento (entrada + forward)')
        eje.plot(posiciones, grupo['pico_vram_mib'], 'ko--', markersize=4, label='Pico del forward')
        eje.set(title=f'{nombre} — CUDA, {nombre_precision}',
                xlabel='Longitud de secuencia (tokens)', ylabel='Memoria asignada (MiB)')
        eje.legend(fontsize=8)
        eje.grid(axis='y', alpha=0.25)
    figura.tight_layout()
    figura.savefig(resultados / 'memoria.png')
    figura.savefig(resultados / 'memoria.pdf')
    plt.show()
else:
    print('VRAM no disponible: la medición se ha realizado en CPU.')